In [1]:
# Import and initialize poolparty
import poolparty as pp 
pp.init()

# Add highlighting specifications
pp.add_highlight(which='tags', style='gray')
pp.add_highlight(region='cre', style='lightskyblue')
pp.add_highlight(which='lower gap', style='bold yellow')

# Step 1: Create template Pool
template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc/>AGATCGGA').named('template_pool')
template_pool.print_library()

template_pool: seq_length=51, num_states=1
state  seq
    0  TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc/>AGATCGGA



Pool(id=0, name='template_pool', op='op[0]:from_seq', num_states=1)

In [2]:
# Step 2: Randomly mutate <cre> region at 20% per nucleotide
# Step 3: Repeat each mutant 3 times
# Step 4: Insert random barcodes at <bc/>
mutagenized_bc_pool = template_pool\
    .mutagenize(region='cre', mutation_rate=0.2, mode='hybrid', num_hybrid_states=5, seq_name_prefix='mut_')\
    .repeat_states(3, seq_name_prefix='v', op_iter_order=-1)\
    .insert_kmers('bc', length=5).named('mutagenized_bc_pool')
mutagenized_bc_pool.print_library()

mutagenized_bc_pool: seq_length=56, num_states=15
state  name      seq
    0  mut_0.v0  TCCCGACT<cre>GtAAAtgGGGCAcTGAGCAatCAGaAcA</cre>ATTACGG<bc>tggcc</bc>AGATCGGA
    1  mut_0.v1  TCCCGACT<cre>GtAAAtgGGGCAcTGAGCAatCAGaAcA</cre>ATTACGG<bc>aaaat</bc>AGATCGGA
    2  mut_0.v2  TCCCGACT<cre>GtAAAtgGGGCAcTGAGCAatCAGaAcA</cre>ATTACGG<bc>gtggt</bc>AGATCGGA
    3  mut_1.v0  TCCCGACT<cre>GGAAAtCGGcCAtTGAGtACAtgtaAAA</cre>ATTACGG<bc>ggggt</bc>AGATCGGA
    4  mut_1.v1  TCCCGACT<cre>GGAAAtCGGcCAtTGAGtACAtgtaAAA</cre>ATTACGG<bc>ctgac</bc>AGATCGGA
    5  mut_1.v2  TCCCGACT<cre>GGAAAtCGGcCAtTGAGtACAtgtaAAA</cre>ATTACGG<bc>tgatg</bc>AGATCGGA
    6  mut_2.v0  TCCCGACT<cre>GGAccGgGGGCgtTGAGCACgtAcGAAA</cre>ATTACGG<bc>taata</bc>AGATCGGA
    7  mut_2.v1  TCCCGACT<cre>GGAccGgGGGCgtTGAGCACgtAcGAAA</cre>ATTACGG<bc>gaccc</bc>AGATCGGA
    8  mut_2.v2  TCCCGACT<cre>GGAccGgGGGCgtTGAGCACgtAcGAAA</cre>ATTACGG<bc>caaaa</bc>AGATCGGA
    9  mut_3.v0  TCCCGACT<cre>GGAAcGgGGGCAGTatGCACACtacAgA</cre>ATTACGG<bc>gggcg</b

Pool(id=3, name='mutagenized_bc_pool', op='op[3]:get_kmers', num_states=15)

In [16]:
pp.clear_pools()

template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc/>AGATCGGA').named('template_pool')

mutated_pool = template_pool.mutagenize('cre',
                                        mutation_rate=0.1, 
                                        mode='hybrid', 
                                        num_hybrid_states=5, 
                                        seq_name_prefix='mut_').named('mutated_pool')

deletion_pool = template_pool.deletion_scan('cre', 
                                            deletion_length=5, 
                                            positions=slice(None, None, 5), 
                                            mode='sequential', 
                                            seq_name_prefix='del_').named('deletion_pool')

sites_pool=pp.from_seqs(['AAAAA','TTTTT'], 
                        seq_names = ['site_a', 'site_b'],
                        mode='sequential', 
                        op_iter_order=-1).named('sites_pool')

insertion_pool = template_pool.insertion_scan('cre', 
                                              ins_pool=sites_pool, 
                                              positions=slice(None,None,5), 
                                              replace=True, 
                                              mode='sequential', 
                                              seq_name_prefix='ins_').named('insertion_pool')

combo_pool = pp.stack([mutated_pool, deletion_pool, insertion_pool])\
    .repeat_states(2, seq_name_prefix='v', op_iter_order=-2)\
    .insert_kmers('bc', length=5)\
    .named('combo_pool')\

combo_pool.print_library()

combo_pool: seq_length=None, num_states=40
state  name             seq
    0  mut_0.v0         TCCCGACT<cre>GGAAAGaaGGCAGgGAGCACAaAaGAAA</cre>ATTACGG<bc>tggcc</bc>AGATCGGA
    1  mut_0.v1         TCCCGACT<cre>GGAAAGaaGGCAGgGAGCACAaAaGAAA</cre>ATTACGG<bc>aaaat</bc>AGATCGGA
    2  mut_1.v0         TCCCGACT<cre>GGAAAGgGGGCAGTGAGCACgCAGGAtc</cre>ATTACGG<bc>gtggt</bc>AGATCGGA
    3  mut_1.v1         TCCCGACT<cre>GGAAAGgGGGCAGTGAGCACgCAGGAtc</cre>ATTACGG<bc>ggggt</bc>AGATCGGA
    4  mut_2.v0         TCCCGACT<cre>GGAAAGCaGGCAGTtAGCACACAaaAcA</cre>ATTACGG<bc>ctgac</bc>AGATCGGA
    5  mut_2.v1         TCCCGACT<cre>GGAAAGCaGGCAGTtAGCACACAaaAcA</cre>ATTACGG<bc>tgatg</bc>AGATCGGA
    6  mut_3.v0         TCCCGACT<cre>GGAAtGCGGGCAGTGAaatCACAGGgAA</cre>ATTACGG<bc>taata</bc>AGATCGGA
    7  mut_3.v1         TCCCGACT<cre>GGAAtGCGGGCAGTGAaatCACAGGgAA</cre>ATTACGG<bc>gaccc</bc>AGATCGGA
    8  mut_4.v0         TCCCGACT<cre>GGgAAGCGGGCtGTGAGCACACAGGAAA</cre>ATTACGG<bc>caaaa</bc>AGATCGGA
    9  mut_4.v1     

Pool(id=10, name='combo_pool', op='op[10]:get_kmers', num_states=40)

In [5]:
combo_pool.print_dag()

combo_pool (pool, n=40)
└── op[10]:get_kmers [mode=random, n=1]
    └── pool[9] (pool, n=40)
        └── op[9]:repeat [mode=sequential, n=2]
            └── pool[8] (pool, n=20)
                └── op[8]:stack [mode=fixed, n=1]
                    ├── mutated_pool (pool, n=5)
                    │   └── op[1]:mutagenize [mode=hybrid, n=5]
                    │       └── template_pool (pool, n=1)
                    │           └── op[0]:from_seq [mode=fixed, n=1]
                    ├── deletion_pool (pool, n=5)
                    │   └── op[3]:deletion_scan(from_seq) [mode=fixed, n=1]
                    │       └── pool[2] (pool, n=5)
                    │           └── op[2]:deletion_scan(marker_scan) [mode=sequential, n=5]
                    │               └── template_pool (pool, n=1)
                    │                   └── op[0]:from_seq [mode=fixed, n=1]
                    └── insertion_pool (pool, n=10)
                        └── op[7]:insertion_scan(replace_marker_con

Pool(id=10, name='combo_pool', op='op[10]:get_kmers', num_states=40)

In [6]:
df = combo_pool.generate_library(report_design_cards=True)
df.columns

Index(['name', 'seq', 'combo_pool.seq', 'pool[9].seq', 'pool[8].seq',
       'deletion_pool.seq', 'insertion_pool.seq', 'pool[5].seq',
       'sites_pool.seq', 'pool[6]:insertion_scan(intermediate).seq',
       'pool[2].seq', 'mutated_pool.seq', 'template_pool.seq',
       'combo_pool.state', 'pool[9].state', 'pool[8].state',
       'deletion_pool.state', 'insertion_pool.state', 'pool[5].state',
       'sites_pool.state', 'pool[6]:insertion_scan(intermediate).state',
       'pool[2].state', 'mutated_pool.state', 'template_pool.state',
       'op[10]:get_kmers.state', 'op[9]:repeat.state', 'op[8]:stack.state',
       'op[3]:deletion_scan(from_seq).state',
       'op[7]:insertion_scan(replace_marker_content).state',
       'op[5]:insertion_scan(swapcase).state', 'op[4]:from_seqs.state',
       'op[6]:insertion_scan(marker_scan).state',
       'op[2]:deletion_scan(marker_scan).state', 'op[1]:mutagenize.state',
       'op[0]:from_seq.state', 'op[10]:get_kmers.key.kmer_index',
       'op[10

In [7]:
renamed_cols = {
    'name': 'name',
    'op[10]:get_kmers.key.kmer': 'bc_seq',
    'op[9]:repeat.state': 'v',
    'op[1]:mutagenize.key.positions': 'mut_positions',
    'op[1]:mutagenize.key.wt_chars': 'mut_wt_chars',
    'op[1]:mutagenize.key.mut_chars': 'mut_mut_chars',
    'op[2]:deletion_scan(marker_scan).key.start': 'del_start',
    'op[2]:deletion_scan(marker_scan).key.stop': 'del_stop',
    #'op[2]:deletion_scan(marker_scan).key.length': 'del_length',
    'op[2]:deletion_scan(marker_scan).key.region_content': 'deleted_seq',
    'op[6]:insertion_scan(marker_scan).key.start': 'ins_start',
    'op[6]:insertion_scan(marker_scan).key.stop': 'ins_stop',
    #'op[6]:insertion_scan(marker_scan).key.length': 'ins_length',
    'op[6]:insertion_scan(marker_scan).key.strand': 'ins_strand',
    'op[6]:insertion_scan(marker_scan).key.region_content': 'replaced_seq',
    'op[4]:from_seqs.key.seq_name': 'site_name',
    'sites_pool.seq': 'site_seq',
}
tmp_df = df.rename(columns=renamed_cols)[renamed_cols.values()]
display(tmp_df)

,name,bc_seq,v,mut_positions,mut_wt_chars,mut_mut_chars,del_start,del_stop,deleted_seq,ins_start,ins_stop,ins_strand,replaced_seq,site_name,site_seq
0,mut_0.v0,TGGCC,0,"(6, 7, 13, 21, 23)","(C, G, T, C, G)","(a, a, g, a, a)",None,None,None,None,None,None,None,None,None
1,mut_0.v1,AAAAT,1,"(6, 7, 13, 21, 23)","(C, G, T, C, G)","(a, a, g, a, a)",None,None,None,None,None,None,None,None,None
2,mut_1.v0,GTGGT,0,"(6, 20, 26, 27)","(C, A, A, A)","(g, g, t, c)",None,None,None,None,None,None,None,None,None
3,mut_1.v1,GGGGT,1,"(6, 20, 26, 27)","(C, A, A, A)","(g, g, t, c)",None,None,None,None,None,None,None,None,None
4,mut_2.v0,CTGAC,0,"(7, 14, 23, 24, 26)","(G, G, G, G, A)","(a, t, a, a, c)",None,None,None,None,None,None,None,None,None
5,mut_2.v1,TGATG,1,"(7, 14, 23, 24, 26)","(G, G, G, G, A)","(a, t, a, a, c)",None,None,None,None,None,None,None,None,None
6,mut_3.v0,TAATA,0,"(4, 16, 17, 18, 25)","(A, G, C, A, A)","(t, a, a, t, g)",None,None,None,None,None,None,None,None,None
7,mut_3.v1,GACCC,1,"(4, 16, 17, 18, 25)","(A, G, C, A, A)","(t, a, a, t, g)",None,None,None,None,None,None,None,None,None
8,mut_4.v0,CAAAA,0,"(2, 11)","(A, A)","(g, t)",None,None,None,None,None,None,None,None,None
9,mut_4.v1,GGGCG,1,"(2, 11)","(A, A)","(g, t)",None,None,None,None,None,None,None,None,None


In [8]:
df = combo_pool.clear_annotation()\
    .generate_library(report_design_cards=False)
df.to_excel('~/Desktop/library.xlsx', index=False)

In [9]:
combo_pool.print_dag()

combo_pool (pool, n=40)
└── op[10]:get_kmers [mode=random, n=1]
    └── pool[9] (pool, n=40)
        └── op[9]:repeat [mode=sequential, n=2]
            └── pool[8] (pool, n=20)
                └── op[8]:stack [mode=fixed, n=1]
                    ├── mutated_pool (pool, n=5)
                    │   └── op[1]:mutagenize [mode=hybrid, n=5]
                    │       └── template_pool (pool, n=1)
                    │           └── op[0]:from_seq [mode=fixed, n=1]
                    ├── deletion_pool (pool, n=5)
                    │   └── op[3]:deletion_scan(from_seq) [mode=fixed, n=1]
                    │       └── pool[2] (pool, n=5)
                    │           └── op[2]:deletion_scan(marker_scan) [mode=sequential, n=5]
                    │               └── template_pool (pool, n=1)
                    │                   └── op[0]:from_seq [mode=fixed, n=1]
                    └── insertion_pool (pool, n=10)
                        └── op[7]:insertion_scan(replace_marker_con

Pool(id=10, name='combo_pool', op='op[10]:get_kmers', num_states=40)

In [10]:
combo_pool.counter.print_dag()

combo_pool.state (counter, io=-2, n=40)
└── [op=Product]
    ├── op[9]:repeat.state (counter, io=-2, n=2)
    ├── pool[8].state (counter, io=-1, n=20)
    │   └── [op=Stack]
    │       ├── mutated_pool.state (counter, io=0, n=5)
    │       │   └── [op=Product]
    │       │       ├── op[1]:mutagenize.state (counter, io=0, n=5)
    │       │       └── template_pool.state (counter, io=0, n=1)
    │       │           └── [op=Passthrough]
    │       │               └── op[0]:from_seq.state (counter, io=0, n=1)
    │       ├── deletion_pool.state (counter, io=0, n=5)
    │       │   └── [op=Product]
    │       │       ├── op[3]:deletion_scan(from_seq).state (counter, io=0, n=1)
    │       │       ├── op[2]:deletion_scan(marker_scan).state (counter, io=0, n=5)
    │       │       └── template_pool.state (counter, io=0, n=1)
    │       │           └── [op=Passthrough]
    │       │               └── op[0]:from_seq.state (counter, io=0, n=1)
    │       └── insertion_pool.state (counter,

In [11]:
counter_df = combo_pool.counter.get_iteration_df()
counter_df.to_excel('~/Desktop/counter_states.xlsx')

In [12]:
import statecounter as sc
A = sc.Counter(3, name='A')
B = sc.Counter(4, name='B')
C = sc.product([A,B], name='C')
C.get_iteration_df().T


C,0,1,2,3,4,5,6,7,8,9,10,11
A,0,1,2,0,1,2,0,1,2,0,1,2
B,0,0,0,1,1,1,2,2,2,3,3,3


In [13]:
import statecounter as sc
A = sc.Counter(3, name='A')
B = sc.Counter(4, name='B')
C = sc.stack([A,B], name='C')
C.get_iteration_df().T

C,0,1,2,3,4,5,6
A,0,1,2,None,None,None,None
B,None,None,None,0,1,2,3


In [14]:
import statecounter as sc
A = sc.Counter(2,name='A')
C = sc.repeat(A,3,name='C')
C.get_iteration_df().T

C,0,1,2,3,4,5
A,0,1,0,1,0,1


In [15]:
import statecounter as sc
A = sc.Counter(12,name='A')
C = sc.slice(A,None,None,3,name='C')
C.get_iteration_df().T

C,0,1,2,3
A,0,3,6,9
